# Demo: Running Time-Series Diagnostics

Fitting a model is only half the job. The other half is checking whether the model actually captured the signal in the data. If the residuals still have structure -- patterns the model missed -- your forecasts and prediction intervals can't be trusted.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

In [ ]:
# Setup: fit both models so we can compare their diagnostics
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date").asfreq("MS")
train = df.loc[:"2023-12-01"]

arima = SARIMAX(train["demand_mwh"], order=(2, 1, 1)).fit(disp=False)
sarimax = SARIMAX(train["demand_mwh"], exog=train["avg_temp_f"], order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
print("Models fitted.")

## What Good Residuals Look Like

If a model is doing its job, the residuals should look like random noise -- no trends, no repeating patterns, just unpredictable jitter around zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(arima.resid, linewidth=0.5)
axes[0].axhline(y=0, color="gray", linestyle="--")
axes[0].set_title("ARIMA residuals")
axes[1].plot(sarimax.resid, linewidth=0.5)
axes[1].axhline(y=0, color="gray", linestyle="--")
axes[1].set_title("SARIMAX residuals")
plt.tight_layout()
plt.show()

## ACF of Residuals

If there's still autocorrelation in the residuals, the model left patterns on the table. We check with the ACF.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(arima.resid, ax=axes[0], lags=24)
axes[0].set_title("ARIMA residual ACF")
plot_acf(sarimax.resid, ax=axes[1], lags=24)
axes[1].set_title("SARIMAX residual ACF")
plt.tight_layout()
plt.show()

## Ljung-Box Test

The ACF gives you a visual, but the Ljung-Box test gives you a number. It tests whether there's significant autocorrelation at a given lag. p-value above 0.05 means the residuals look like white noise -- that's what we want.

In [ ]:
for name, fitted in [("ARIMA", arima), ("SARIMAX", sarimax)]:
    lb = acorr_ljungbox(fitted.resid, lags=[12], return_df=True)
    _, norm_p = stats.normaltest(fitted.resid.dropna())
    print(f"{name}:")
    print(f"  AIC: {fitted.aic:.1f}  BIC: {fitted.bic:.1f}")
    print(f"  Ljung-Box p (lag 12): {lb['lb_pvalue'].values[0]:.4f}")
    print(f"  Normality p: {norm_p:.4f}")
    print()

Lower AIC and BIC means a better model. The Ljung-Box p-value tells you if there's remaining autocorrelation. The normality p-value tells you if the prediction intervals can be trusted -- if normality fails, the intervals are still directionally useful but the exact percentages are unreliable.

SARIMAX wins on information criteria because temperature actually explains variance. Whether it also has cleaner residuals depends on your specific data -- that's what diagnostics are for.